# SparkClient: Apache Iceberg with MinIO

This notebook creates a Spark Connect session with Apache Iceberg and a MinIO S3-compatible object store. It creates an Iceberg table, writes sample rows, reads them back, and inspects the table metadata.

The Spark cluster must be able to reach both MinIO and the Iceberg REST catalog. For the repository's Kind setup, create the cluster with `make test-e2e-setup-cluster`, then install the optional services with `make test-e2e-setup-fixtures`.

Set `MINIO_ENDPOINT`, `ICEBERG_REST_URI`, and the credential environment variables when using a different setup.


## 1. Configure the Spark client

Use the namespace provided by the E2E workflow when available. Local defaults are intended for development only.

In [ ]:
import os
import uuid

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import Name, SparkClient

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
minio_endpoint = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
minio_secret_key = os.environ.get("MINIO_SECRET_KEY", "minioadmin")
iceberg_rest_uri = os.environ.get("ICEBERG_REST_URI", "http://iceberg-rest:8181")
warehouse = os.environ.get("ICEBERG_WAREHOUSE", "s3://warehouse/")


backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)
session_name = f"kubeflow-iceberg-minio-{uuid.uuid4().hex[:8]}"

print(f"Using namespace: {namespace}")
print(f"MinIO endpoint: {minio_endpoint}")
print(f"Iceberg REST catalog: {iceberg_rest_uri}")

## 2. Start an Iceberg-enabled Spark Connect session

The Iceberg runtime and AWS bundle are loaded directly from Maven. The catalog uses `S3FileIO` with path-style access for MinIO.

In [ ]:
iceberg_version = "1.11.0"
maven = "https://repo1.maven.org/maven2/org/apache/iceberg"

spark = client.connect(
    spark_conf={
        "spark.jars": ",".join(
            [
                f"{maven}/iceberg-spark-runtime-4.0_2.13/{iceberg_version}/"
                f"iceberg-spark-runtime-4.0_2.13-{iceberg_version}.jar",
                f"{maven}/iceberg-aws-bundle/{iceberg_version}/"
                f"iceberg-aws-bundle-{iceberg_version}.jar",
            ]
        ),
        "spark.sql.extensions": (
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
        ),
        "spark.sql.catalog.lakehouse": "org.apache.iceberg.spark.SparkCatalog",
        "spark.sql.catalog.lakehouse.type": "rest",
        "spark.sql.catalog.lakehouse.uri": iceberg_rest_uri,
        "spark.sql.catalog.lakehouse.warehouse": warehouse,
        "spark.sql.catalog.lakehouse.io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
        "spark.sql.catalog.lakehouse.s3.endpoint": minio_endpoint,
        "spark.sql.catalog.lakehouse.s3.path-style-access": "true",
        "spark.sql.catalog.lakehouse.client.region": "us-east-1",
        "spark.sql.catalog.lakehouse.s3.access-key-id": minio_access_key,
        "spark.sql.catalog.lakehouse.s3.secret-access-key": minio_secret_key,
    },
    options=[Name(session_name)],
)
print(f"Connected Spark session: {session_name}")

## 3. Create and populate an Iceberg table

The table is reset before writing so that rerunning this notebook produces the same two rows.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.demo")
spark.sql("DROP TABLE IF EXISTS lakehouse.demo.users")
spark.sql(
    """
    CREATE TABLE lakehouse.demo.users (
        id BIGINT,
        name STRING
    ) USING iceberg
    """
)
spark.sql("INSERT INTO lakehouse.demo.users VALUES (1, 'Alice'), (2, 'Bob')")

users = spark.sql("SELECT * FROM lakehouse.demo.users ORDER BY id")
users.show()
rows = [tuple(row) for row in users.collect()]
assert rows == [(1, "Alice"), (2, "Bob")], rows

## 4. Inspect table metadata

In [ ]:
snapshots = spark.sql("SELECT snapshot_id, operation FROM lakehouse.demo.users.snapshots").collect()
print(f"Snapshots: {[(s.snapshot_id, s.operation) for s in snapshots]}")
assert len(snapshots) == 1 and snapshots[0].operation == "append", snapshots

data_files = [
    row.file_path for row in spark.sql("SELECT file_path FROM lakehouse.demo.users.files").collect()
]
print("Data files:")
for path in data_files:
    print(f"  {path}")
assert data_files, "Expected at least one data file"
assert all(path.startswith(warehouse) for path in data_files), data_files

## 5. Clean up the Spark session

Stop the local Spark Connect client and remove its session resource from Kubernetes.

In [ ]:
spark.sql("DROP TABLE IF EXISTS lakehouse.demo.users PURGE")
spark.stop()
client.delete_session(session_name)
print("Iceberg + MinIO example complete.")